# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mowleen12/flyrank-ml-project-1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup: Load Research Paper/Dataset

First, we'll install the `datasets` library to access the provided Hugging Face dataset, which should contain the research paper or related information for this assignment.

In [1]:
# Install the Hugging Face datasets library
!pip install datasets

In [2]:
from datasets import load_dataset

# Load the dataset from the provided link
dataset_name = 'FlyRank/internship-starter'
dataset = load_dataset(dataset_name)

print("Dataset loaded successfully:")
print(dataset)

# Inspect a sample from the dataset to understand its structure and contents
# This might reveal how the 'research paper' is included or referenced.
if 'train' in dataset:
    print("\nSample from the 'train' split:")
    print(dataset['train'][0])
elif len(dataset) > 0:
    # Get the first split if 'train' is not available
    first_split_name = list(dataset.keys())[0]
    print(f"\nSample from the '{first_split_name}' split:")
    print(dataset[first_split_name][0])
else:
    print("\nNo splits found in the dataset.")

README.md:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

content_refresh_anonymized.csv:   0%|          | 0.00/8.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30000 [00:00<?, ? examples/s]

Dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'health_score', 'needs_indexing', 'is_quick_win', 'needs_ctr_fix', 'needs_engagement_fix', 'ai_opportunity', 'is_underperformer', 'is_declining', '

In [4]:
# Inspect the dataset info for a description or potential research findings
print("\nDataset Info Description:")
if 'train' in dataset:
    print(dataset['train'].info.description)
else:
    print("No 'train' split found to get description from.")

print("\nDataset Info Features:")
if 'train' in dataset:
    print(dataset['train'].info.features)
else:
    print("No 'train' split found to get features from.")

# Let's also try to load the dataset card markdown to see if there's a more detailed 'paper' content
# This might require a separate call or direct fetching from Hugging Face if not directly in dataset.info
from huggingface_hub import HfApi

api = HfApi()
repo_id = 'FlyRank/internship-starter'

try:
    # Get the dataset card content (README.md)
    readme_content = api.dataset_info(repo_id, raw_for_description=True).cardData['text']
    print("\n--- README.md Content ---")
    print(readme_content)
    print("\n--------------------------")
except Exception as e:
    print(f"Could not load README.md content programmatically: {e}")
    print("You may need to manually review the dataset card on Hugging Face to find the research paper findings.")


Dataset Info Description:


Dataset Info Features:
{'content_id': Value('string'), 'client_id': Value('string'), 'search_volume': Value('float64'), 'competition': Value('float64'), 'competition_level': Value('string'), 'cpc': Value('float64'), 'content_type': Value('string'), 'main_intent': Value('string'), 'word_count': Value('float64'), 'char_count': Value('float64'), 'provider_used': Value('string'), 'model_used': Value('string'), 'impressions_90d': Value('int64'), 'clicks_90d': Value('int64'), 'pageviews_90d': Value('int64'), 'sessions_90d': Value('int64'), 'users_90d': Value('int64'), 'engaged_sessions_90d': Value('int64'), 'ai_sessions_90d': Value('int64'), 'scroll_events_90d': Value('int64'), 'days_with_impressions': Value('int64'), 'days_with_sessions': Value('int64'), 'impressions_last_30d': Value('int64'), 'clicks_last_30d': Value('int64'), 'sessions_last_30d': Value('int64'), 'impressions_prev_30d': Value('int64'), 'clicks_prev_30d': Value('int64'), 'sessions_prev_30d': Val

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Given the rich set of features in the dataset related to content performance and optimization, and the inability to programmatically access a specific research paper text from the Hugging Face dataset directly, I will propose two plausible findings that such a paper might investigate, based on the available data:

### Finding 1: Content with a 'declining' trend is more likely to be an 'initial refresh candidate'.

*   **Where does the label come from?**
    *   `trend_direction`: This feature likely indicates whether a content piece's performance (e.g., impressions, clicks) is increasing, stable, or decreasing over a recent period. It's an observed metric calculated from time-series data. The 'declining' label would be derived from this. \n
    *   `is_initial_refresh_candidate`: This is a boolean flag. It likely represents a classification or recommendation by an internal model or heuristic at FlyRank, identifying content segments that are prime candidates for a refresh or update. This could be an output of a predictive model or a business rule.

*   **Does the validation design carry the claim?**
    *   To validate this claim, one would typically need to perform a study where content identified as 'declining' and 'initial refresh candidate' is actually refreshed, and its subsequent performance is compared against a control group of similar content that was not refreshed. The dataset provides `impressions_90d`, `clicks_90d`, `sessions_90d`, etc., which could serve as outcome variables. However, without knowing if refreshes were actually performed and tracked for this dataset, or if the `is_initial_refresh_candidate` label was used as a treatment, it's hard to definitively say if the validation design (as implied by the dataset structure) *alone* carries the claim. The existing data allows for correlation analysis (e.g., what percentage of declining content is marked as an initial refresh candidate?), but a true causal claim requires an experimental setup or a robust quasi-experimental design.


### Finding 2: Content with a lower 'health_score' is more likely to 'need a CTR fix' or 'engagement fix'.

*   **Where does the label come from?**
    *   `health_score`: This is an integer value, likely a composite metric or an output from an internal model that aggregates various performance indicators (e.g., engagement rate, CTR, trend direction) into a single score representing content health. It's a derived, evaluative metric.

    *   `needs_ctr_fix` and `needs_engagement_fix`: These are boolean flags. They likely indicate that a content piece has been identified as underperforming specifically in Click-Through Rate (CTR) or engagement metrics (e.g., scroll rate, session duration), perhaps falling below certain thresholds. These could be outputs of diagnostic models or rule-based systems.

*   **Does the validation design carry the claim?**
    *   This claim implies a diagnostic relationship: a low health score *points to* specific performance issues. Validation would involve showing that content identified by a low `health_score` and `needs_ctr_fix`/`needs_engagement_fix` truly has measurable low CTR or engagement, and that interventions targeting these specific fixes (e.g., A/B testing new headlines for `needs_ctr_fix` content, or improving content structure for `needs_engagement_fix` content) lead to improvements. The dataset provides `ctr`, `engagement_rate`, and `scroll_rate` as direct metrics, allowing for an audit of how `health_score`, `needs_ctr_fix`, and `needs_engagement_fix` correlate with these actual performance metrics. The dataset allows for assessing the *accuracy* of these diagnostic flags (e.g., does content with `needs_ctr_fix=True` actually have a low CTR?), but a true validation of the *effectiveness of fixing* these issues would require tracking post-intervention performance, which isn't explicitly available in the dataset for a causal inference.

## 2. My model under an honest split (before/after)

To address this section, I will build a simple classification model to predict whether a content piece `is_initial_refresh_candidate`. This target variable is a boolean, making it suitable for binary classification. I will implement a time-aware split based on `content_age_days` to simulate a realistic scenario where the model predicts on newer content after being trained on older content. A `LogisticRegression` model will be used for its interpretability and simplicity.


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Convert the 'train' split of the dataset to a pandas DataFrame
df = dataset['train'].to_pandas()

# Define target and features
target = 'is_initial_refresh_candidate'

# Selecting features that would plausibly be available at the time of prediction
# and are not direct future outcomes or leak information.
# Some performance metrics are included as they reflect past performance.
features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
    'ai_traffic_pct', 'health_score',
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier',
    'position_tier', 'trend_direction'
]

# Separate features (X) and target (y)
X = df[features].copy() # Use .copy() to avoid SettingWithCopyWarning
y = df[target].astype(bool) # Explicitly convert target to boolean

# Handle missing values by simple imputation for numerical features and 'Unknown' for categorical
# For simplicity, we'll fill numerical NaNs with the mean and categorical with a placeholder
for col in X.select_dtypes(include=['float64', 'int64']).columns:
    if X[col].isnull().any():
        X.loc[:, col] = X[col].fillna(X[col].mean())

for col in X.select_dtypes(include=['object']).columns:
    if X[col].isnull().any():
        X.loc[:, col] = X[col].fillna('Unknown')

# Identify numerical and categorical features
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Create the pipeline with preprocessing and logistic regression model
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', LogisticRegression(solver='liblinear', random_state=42))])

# Time-aware split: Split data based on 'content_age_days'
# Let's say content older than the 70th percentile is for training, newer for testing
split_threshold = df['content_age_days'].quantile(0.7)

X_train_time = X[df['content_age_days'] <= split_threshold]
y_train_time = y[df['content_age_days'] <= split_threshold]
X_test_time = X[df['content_age_days'] > split_threshold]
y_test_time = y[df['content_age_days'] > split_threshold]

print(f"Time-aware split: Training on content with age <= {split_threshold:.2f} days, Testing on content with age > {split_threshold:.2f} days")
print(f"Training set size: {len(X_train_time)}")
print(f"Testing set size: {len(X_test_time)}")

# Train the model on the time-aware training data
model_pipeline.fit(X_train_time, y_train_time)

# Make predictions on the training and test sets
y_train_pred = model_pipeline.predict(X_train_time)
y_test_pred = model_pipeline.predict(X_test_time)

# Calculate evaluation metrics
def evaluate_model(y_true, y_pred, name=""):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"{name} Metrics:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")

# Evaluate on training set
print("\n--- Performance on 'Before' (Training) Data ---")
evaluate_model(y_train_time, y_train_pred, "Training")

# Evaluate on testing set
print("\n--- Performance on 'After' (Testing) Data ---")
evaluate_model(y_test_time, y_test_pred, "Testing")

Time-aware split: Training on content with age <= 321.00 days, Testing on content with age > 321.00 days
Training set size: 21040
Testing set size: 8960

--- Performance on 'Before' (Training) Data ---
Training Metrics:
  Accuracy: 0.8390
  Precision: 0.7675
  Recall: 0.7484
  F1-Score: 0.7579

--- Performance on 'After' (Testing) Data ---
Testing Metrics:
  Accuracy: 0.6444
  Precision: 0.5753
  Recall: 0.9926
  F1-Score: 0.7284


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The model was trained on 'older' content (content_age_days less than or equal to the 70th percentile) and evaluated on 'newer' content (content_age_days greater than the 70th percentile). This simulates a more realistic deployment scenario where the model predicts on future data. The metrics show a drop in performance from the training set to the test set, which is expected due to the temporal shift. This suggests that factors influencing 'is_initial_refresh_candidate' might change over time, or the model is overfitting to the older data. More advanced time-series modeling or feature engineering could be considered to improve generalization on newer data.


## 3. Leakage audit

Data leakage occurs when information that would not be available at prediction time is used during model training. This can lead to overly optimistic performance estimates. Here, I'll audit the feature set (`features`) used to predict `is_initial_refresh_candidate`.

**Definition of Leakage for this context:** A feature is considered leaky if its value directly or indirectly incorporates information from the target variable that would not be known at the moment a real-time prediction for `is_initial_refresh_candidate` is needed. Given that `is_initial_refresh_candidate` is a boolean flag indicating a content piece is a *candidate* for refresh, features representing future outcomes *after* this candidacy decision would be leaky. Features representing past or concurrent states are generally acceptable.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [7]:
print("Features used in the model:")
for feature in features:
    if feature == 'health_score':
        print(f"- {feature}: (Potential Leakage) This is a composite score. If its calculation incorporates criteria that are directly or causally linked to the 'is_initial_refresh_candidate' label (e.g., if content marked as a refresh candidate automatically gets a lower health score, or if factors only known after the candidacy decision influence it), it could be leaky. Assuming it reflects overall past performance and intrinsic content quality up to the point of decision, it's a valid input. A deeper understanding of its calculation is needed.")
    elif feature == 'trend_direction':
        print(f"- {feature}: (Potential Leakage) Similar to 'health_score', if 'trend_direction' is determined by performance metrics that extend *beyond* the point of refresh candidacy identification, it could be leaky. However, if it represents the trend *leading up to* the candidacy decision, it's a valid predictive feature.")
    elif feature.endswith('_90d') or feature.endswith('_30d'):
        print(f"- {feature}: (Generally Not Leaky) These aggregated performance metrics (e.g., impressions_90d, clicks_last_30d) represent historical performance. As long as the aggregation window does not extend into the 'future' relative to the point when 'is_initial_refresh_candidate' is determined, these are valid features representing past content performance.")
    elif feature == 'content_age_days' or feature == 'days_since_last_update':
        print(f"- {feature}: (Not Leaky) These are temporal attributes of the content itself, available at any given time.")
    elif feature in ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']:
        print(f"- {feature}: (Not Leaky) These are categorical descriptions or tiers derived from other features or content attributes, generally available at the time of prediction.")
    else:
        print(f"- {feature}: (Not Leaky) These are intrinsic content properties or directly observed metrics (e.g., search_volume, cpc, word_count, ctr, avg_position), generally available at the time of prediction.")

print("\nSummary of Leakage Audit:\nThe primary concerns for potential leakage lie in features like `health_score` and `trend_direction` if their computation implicitly includes future information or decision-making about refresh candidacy. Without explicit timestamps or definitions of how these are derived relative to the target, a definitive conclusion is difficult. The time-aware split helps mitigate some temporal leakage, but direct informational leakage from feature definition remains a possibility. For the purpose of this exercise, we assume these are derived from historical data up to the decision point.")

Features used in the model:
- search_volume: (Not Leaky) These are intrinsic content properties or directly observed metrics (e.g., search_volume, cpc, word_count, ctr, avg_position), generally available at the time of prediction.
- competition: (Not Leaky) These are intrinsic content properties or directly observed metrics (e.g., search_volume, cpc, word_count, ctr, avg_position), generally available at the time of prediction.
- cpc: (Not Leaky) These are intrinsic content properties or directly observed metrics (e.g., search_volume, cpc, word_count, ctr, avg_position), generally available at the time of prediction.
- word_count: (Not Leaky) These are intrinsic content properties or directly observed metrics (e.g., search_volume, cpc, word_count, ctr, avg_position), generally available at the time of prediction.
- char_count: (Not Leaky) These are intrinsic content properties or directly observed metrics (e.g., search_volume, cpc, word_count, ctr, avg_position), generally available at

## 4. Claim rewrite

**Original Bold Claim (from Section 1, Finding 1):** "Content with a 'declining' trend is more likely to be an 'initial refresh candidate'."

This claim, while based on observation, can be strengthened and made more precise with careful wording. Here's a rewrite incorporating "observed," "measured," "directional," and "decision-support":

**Rewritten Claim:** "Analysis of historical data indicates that content pieces exhibiting a **measured** 'declining' performance **trend direction** are **observed** to have a higher frequency of being classified as an 'initial refresh candidate'. This correlation provides **decision-support** for prioritizing content for review or intervention, suggesting that a declining trend serves as an indicator for potential refresh candidacy, rather than a direct cause."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.